# AI Project Guide — Notebook de puente (Colab / Jupyter)

Prototipa y valida en Python la misma lógica que corre en el navegador
(`src/app.js`), y exporta el JSON de estado en el formato exacto que la
app web puede importar (`docs/architecture/schema-persistencia.json`).

**Reglas:**
- Sin datos reales de la empresa; usa siempre datos sintéticos.
- No se conecta a ninguna API de IA de pago. Toda la lógica es determinística.
- El resultado (`schema_procesos_ia.json`) es el contrato que le pasas a
  Claude Code / Antigravity para construir o ajustar la app.

## 1. Modelos base (sin datos reales)

In [ ]:
from pydantic import BaseModel
from typing import List, Optional, Dict
import json, uuid, datetime

class IdeaPropuesta(BaseModel):
    id_idea: str
    departamento: str
    descripcion_tarea: str
    horas_semanales_estimadas: float
    nivel_autonomia_requerido: str  # 'Presentacion', 'Automatizacion', 'Skill', 'Agente'
    herramientas_involucradas: List[str]

## 2. Motor de clasificación (Sección 3 — nivel de solución)

In [ ]:
def evaluar_caso_de_uso(idea: IdeaPropuesta) -> dict:
    """Clasifica la idea según retorno y complejidad estimada. Reglas fijas, sin IA."""
    puntaje_impacto = idea.horas_semanales_estimadas * 2

    if idea.nivel_autonomia_requerido == "Agente":
        complejidad = "Alta"
    elif idea.nivel_autonomia_requerido in ["Automatizacion", "Skill"]:
        complejidad = "Media"
    else:
        complejidad = "Baja"

    return {
        "id_idea": idea.id_idea,
        "impacto_estimado": puntaje_impacto,
        "complejidad": complejidad,
        "recomendacion_desarrollo": "Prioridad Alta" if puntaje_impacto > 10 and complejidad != "Alta" else "Fase Posterior"
    }

ejemplo = IdeaPropuesta(
    id_idea="IDEA-001", departamento="Operaciones",
    descripcion_tarea="Procesamiento rutinario de reportes periódicos",
    horas_semanales_estimadas=8.5, nivel_autonomia_requerido="Automatizacion",
    herramientas_involucradas=["Excel", "Email"]
)
print(evaluar_caso_de_uso(ejemplo))

## 3. Capacidad 80/20 (Sección 1 — matriz de capacidad)

In [ ]:
def calcular_capacidad_equipo(colaboradores: List[Dict], horas_jornada_semanal: float = 40.0, porcentaje_innovacion: float = 0.20) -> Dict:
    """Aplica la regla 80/20 y clasifica el estado de carga de cada colaborador."""
    horas_operacion_objetivo = horas_jornada_semanal * (1.0 - porcentaje_innovacion)
    horas_innovacion_objetivo = horas_jornada_semanal * porcentaje_innovacion

    total_horas_operativas = 0.0
    total_capacidad_innovacion_disponible = 0.0
    resultados_colaboradores = []

    for persona in colaboradores:
        nombre = persona["nombre"]
        horas_ops = persona["horas_operativas_actuales"]
        total_horas_operativas += horas_ops
        horas_disponibles_totales = max(0.0, horas_jornada_semanal - horas_ops)
        horas_para_ia = min(horas_innovacion_objetivo, horas_disponibles_totales)

        if horas_ops > horas_jornada_semanal:
            estado = "Sobrecargado (Riesgo de Burnout)"
        elif horas_ops > horas_operacion_objetivo:
            estado = "Carga Alta (Capacidad reducida)"
        elif horas_ops == horas_operacion_objetivo:
            estado = "Balance Ideal (80/20 Perfecto)"
        else:
            estado = "Capacidad Ociosa (Ideal para acelerar desarrollos)"

        total_capacidad_innovacion_disponible += horas_para_ia
        resultados_colaboradores.append({
            "colaborador": nombre, "horas_operativas": horas_ops,
            "horas_disponibles_para_ia": horas_para_ia, "estado_carga": estado
        })

    return {
        "resumen_equipo": {
            "total_integrantes": len(colaboradores),
            "horas_operativas_totales": total_horas_operativas,
            "horas_totales_disponibles_para_ia": total_capacidad_innovacion_disponible,
            "porcentaje_carga_operativa_promedio": round((total_horas_operativas / (len(colaboradores) * horas_jornada_semanal)) * 100, 2)
        },
        "detalle_colaboradores": resultados_colaboradores
    }

equipo_ejemplo = [
    {"nombre": "Ana (Analista)", "horas_operativas_actuales": 32.0},
    {"nombre": "Carlos (Coordinador)", "horas_operativas_actuales": 45.0},
    {"nombre": "María (Desarrolladora)", "horas_operativas_actuales": 25.0}
]
print(json.dumps(calcular_capacidad_equipo(equipo_ejemplo), indent=2, ensure_ascii=False))

## 4. Generador del payload de estado (compatible con la app web)

In [ ]:
def nuevo_expediente(nombre_proceso: str, departamento: str) -> Dict:
    """Crea un esqueleto de estado compatible con docs/architecture/schema-persistencia.json"""
    ahora = datetime.datetime.utcnow().isoformat() + "Z"
    return {
        "schema_version": "1.0.0",
        "app_meta": {
            "id_expediente": f"PROC-{uuid.uuid4().hex[:6].upper()}",
            "nombre_proyecto": nombre_proceso,
            "etapa_actual": 1,
            "etapas_completadas": [],
            "generado_en": ahora,
            "actualizado_en": ahora,
            "version_app": "1.0.0"
        },
        "seccion_1_ordenar_trabajo": {
            "metadata_proceso": {
                "id_proceso": nombre_proceso.replace(" ", "-").upper()[:20],
                "nombre_proceso": nombre_proceso,
                "departamento": departamento,
                "responsable_proceso": "",
                "fecha_evaluacion": datetime.date.today().isoformat(),
                "nivel_madurez_actual": "Nivel 1: Asistencia / Presentaciones"
            }
        }
    }

expediente = nuevo_expediente("Conciliación de reportes operativos", "Operaciones")
with open("schema_procesos_ia.json", "w", encoding="utf-8") as f:
    json.dump(expediente, f, indent=2, ensure_ascii=False)
print("Generado schema_procesos_ia.json — cárgalo en src/index.html con 'Cargar expediente'.")

## 5. Puente a Claude Code / Antigravity

1. Descarga `schema_procesos_ia.json` de este notebook.
2. Ábrelo en la app (`src/index.html` → "Cargar expediente") para seguir
   completando las 4 secciones desde el navegador.
3. Cuando necesites ajustar la app o construir la automatización sugerida,
   copia el prompt correspondiente de `docs/guia-prompts-claude.md` y
   pégalo en Claude Code, adjuntando este JSON como contexto — nunca datos
   reales de la empresa, solo esta estructura sintética.